# Gemma 4 Batch Evaluation

This notebook evaluates the Google Gemma 4 model on a dataset of audio segments.
It uses the direct batch inference approach with AutoModelForMultimodalLM.

In [ ]:
#@title Install packages and apply hack
%pip uninstall -y transformers
%pip install -q -U transformers peft accelerate
%pip install -U mistral_common

import builtins

# The Magic Hack: Create a dummy class and inject it into Python's builtins 
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

builtins.PeftConfigLike = DummyPeftConfig

import os
import json
import sys
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM
from google.cloud import storage
from huggingface_hub import login

# Add project root to path
sys.path.append("/app")

# Import common utils
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.eval_runner import run_batch_evaluation

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Configuration ---
HF_TOKEN = "<YOUR_HUGGING_FACE_TOKEN>"
MODEL_NAME = "google/gemma-4-E2B-it"
SELECTED_MODEL_KEY = "gemma-4-E2B-it"

GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"
BATCH_SIZE = 4
LIMIT = 10

# Login to Hugging Face
if HF_TOKEN and not HF_TOKEN.startswith("<"):
    login(token=HF_TOKEN)

print("Imports finally successful!")

In [ ]:
#@title Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Gemma model on {device}...")

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

In [ ]:
#@title Define helper functions for evaluation runner

from pydub import AudioSegment
import os

def prompt_formatter(entry, local_path):
    """Constructs the message structure for Gemma 4."""
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text", 
                    "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three."
                },
                {"type": "audio", "audio": local_path},
            ]
        }
    ]

def gemma_inference(model, prompts):
    """Runs inference using processor and model.generate directly."""
    outputs = []
    for p in prompts:
        audio_path = p[0]["content"][1]["audio"]
        clean_path = audio_path + "_clean.wav"
        
        try:
            # Preprocess audio to 16kHz Mono WAV
            audio = AudioSegment.from_file(audio_path)
            audio = audio.set_frame_rate(16000).set_channels(1)
            audio.export(clean_path, format="wav")
            
            p[0]["content"][1]["audio"] = clean_path
            
            input_ids = processor.apply_chat_template(
                p,
                add_generation_prompt=True,
                tokenize=True, 
                return_dict=True,
                return_tensors="pt",
            )
            input_ids = input_ids.to(model.device, dtype=model.dtype)
            
            out = model.generate(**input_ids, max_new_tokens=100)
            outputs.append(out)
            
        finally:
            if os.path.exists(clean_path):
                os.remove(clean_path)
                
    return outputs

def result_decoder(ans, model):
    """Extracts only the generated transcription from Gemma 4's output."""
    text = processor.batch_decode(
        ans,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False
    )
    
    full_text = text[0]
    
    # Split by the model turn marker for Gemma 4
    if "<|turn>model\n" in full_text:
        answer = full_text.split("<|turn>model\n")[1]
        answer = answer.replace("<turn|>", "").strip()
        return answer
        
    return full_text.strip()

In [ ]:
#@title Run Evaluation

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_batch_evaluation(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=gemma_inference,
    decode_fn=result_decoder,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT
)



In [ ]:
# Upload results directly to GCS from memory (uncomment to use)
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    SELECTED_MODEL_KEY, 
    EXPERIMENT_NAME, 
    results_list
)